In [ ]:

# ===== Deep EVIDENTIAL-regression ENSEMBLE (Amini 2020 NIG) for F2 abstention uncertainty =====
# Outputs calibrated (mean, epistemic, aleatoric) for the 513 test AND CRC-train OOF, so the abstention
# gate can be fit on train-OOF and applied to the 253 with NO leakage. Run on Kaggle T4 (cc7.5).
import numpy as np, pandas as pd, glob, os, time, traceback, torch, torch.nn as nn

def find(name):
    h = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not h:
        for r,_,fs in os.walk('/kaggle/input'):
            for ff in fs: print('  ', os.path.join(r,ff))
        raise FileNotFoundError(name)
    return sorted(h, key=len)[0]

try:
    comp = pd.read_parquet(find('found_compounds.parquet')).sort_values('comp_id').reset_index(drop=True)
    feat = pd.read_parquet(find('found_features.parquet')).sort_values('comp_id').reset_index(drop=True)
    crc  = pd.read_parquet(find('found_crc.parquet'))
    dev = 'cuda' if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 7 else 'cpu'
    print('device', dev, '| compounds', len(comp), '| crc', len(crc))

    X = feat.drop(columns='comp_id').values.astype('float32')
    mu, sd = X.mean(0), X.std(0)+1e-6
    Xall = torch.tensor((X-mu)/sd, device=dev)
    Dh = Xall.shape[1]

    crc_id = crc['comp_id'].astype(int).values
    crc_y  = crc['pec50'].astype('float32').values
    ymu, ysd = crc_y.mean(), crc_y.std()+1e-6
    crc_yz = (crc_y - ymu)/ysd

    n_test = int(comp['test_pos'].max())+1
    tc = np.full(n_test, -1, dtype=int); tm = comp[comp['test_pos']>=0]
    tc[tm['test_pos'].astype(int).values] = tm['comp_id'].astype(int).values

    class Evid(nn.Module):
        def __init__(self, d, p=0.2):
            super().__init__()
            self.body = nn.Sequential(
                nn.Linear(d,1024), nn.BatchNorm1d(1024), nn.ReLU(), nn.Dropout(p),
                nn.Linear(1024,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p),
                nn.Linear(512,256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(p))
            self.out = nn.Linear(256,4)  # gamma, log_nu, log_alpha, log_beta
        def forward(self,x):
            o = self.out(self.body(x))
            gamma = o[:,0]
            nu    = nn.functional.softplus(o[:,1]) + 1e-4
            alpha = nn.functional.softplus(o[:,2]) + 1.0 + 1e-4
            beta  = nn.functional.softplus(o[:,3]) + 1e-4
            return gamma, nu, alpha, beta

    def evid_loss(y, gamma, nu, alpha, beta, lam=0.05):
        Om = 2*beta*(1+nu)
        nll = (0.5*torch.log(np.pi/nu) - alpha*torch.log(Om)
               + (alpha+0.5)*torch.log(nu*(y-gamma)**2 + Om)
               + torch.lgamma(alpha) - torch.lgamma(alpha+0.5))
        reg = torch.abs(y-gamma)*(2*nu+alpha)
        return (nll + lam*reg).mean()

    def train_member(Xtr, ytr, seed, epochs=250):
        torch.manual_seed(seed); np.random.seed(seed)
        m = Evid(Dh).to(dev)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3, weight_decay=1e-5)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
        n = len(ytr); bs = 512
        for ep in range(epochs):
            m.train(); perm = torch.randperm(n, device=dev)
            for b in range(0, n, bs):
                bi = perm[b:b+bs]; opt.zero_grad()
                g,nu,al,be = m(Xtr[bi])
                evid_loss(ytr[bi], g,nu,al,be).backward(); opt.step()
            sch.step()
        m.eval(); return m

    def predict(m, Xq):
        with torch.no_grad():
            g,nu,al,be = m(Xq)
        ale = (be/(al-1)).cpu().numpy()           # aleatoric var
        epi = (be/(nu*(al-1))).cpu().numpy()       # evidential epistemic var
        return g.cpu().numpy(), ale, epi

    K, E = 5, 6   # folds x ensemble members
    rng = np.random.RandomState(0); fold = rng.randint(0, K, size=len(crc_id))
    Xc = Xall[crc_id]
    oof_mean = np.zeros(len(crc_id)); oof_epi = np.zeros(len(crc_id)); oof_ale = np.zeros(len(crc_id))
    test_means = []; test_epis = []; test_ales = []
    Xtest = Xall[np.clip(tc,0,None)]
    t0=time.time()
    for k in range(K):
        tri = np.where(fold!=k)[0]; vai = np.where(fold==k)[0]
        Xtr = Xc[tri]; ytr = torch.tensor(crc_yz[tri], device=dev)
        gms=[]; eps=[]; als=[]; tg=[]; te=[]; ta=[]
        for e in range(E):
            m = train_member(Xtr, ytr, seed=100*k+e)
            g,a,p = predict(m, Xc[vai]); gms.append(g); als.append(a); eps.append(p)
            g2,a2,p2 = predict(m, Xtest); tg.append(g2); ta.append(a2); te.append(p2)
        gms=np.array(gms); eps=np.array(eps); als=np.array(als)
        oof_mean[vai] = gms.mean(0)
        oof_epi[vai]  = gms.var(0) + eps.mean(0)     # ensemble-disagreement + evidential epistemic
        oof_ale[vai]  = als.mean(0)
        tg=np.array(tg); te=np.array(te); ta=np.array(ta)
        test_means.append(tg.mean(0)); test_epis.append(tg.var(0)+te.mean(0)); test_ales.append(ta.mean(0))
        print(f'  fold {k} done ({time.time()-t0:.0f}s)')
    tmean = np.mean(test_means,0); tepi = np.mean(test_epis,0); tale = np.mean(test_ales,0)

    # de-standardize means; uncertainties scaled by ysd^2
    oof = pd.DataFrame({'comp_id':crc_id, 'y':crc_y, 'mean':oof_mean*ysd+ymu,
                        'epistemic':oof_epi*ysd*ysd, 'aleatoric':oof_ale*ysd*ysd})
    test = pd.DataFrame({'test_pos':range(n_test),
                         'mean':np.where(tc>=0, tmean*ysd+ymu, np.nan),
                         'epistemic':np.where(tc>=0, tepi*ysd*ysd, np.nan),
                         'aleatoric':np.where(tc>=0, tale*ysd*ysd, np.nan)})
    oof.to_parquet('/kaggle/working/found_evid_trainoof.parquet', index=False)
    test.to_parquet('/kaggle/working/found_evid_513.parquet', index=False)
    print('saved evid OOF + 513.  OOF RAE(mean vs y) =',
          float(np.mean(np.abs(oof.y-oof['mean']))/np.mean(np.abs(oof.y-oof.y.mean()))))
    print('epi-vs-|err| corr (OOF):', float(np.corrcoef(oof.epistemic, np.abs(oof.y-oof['mean']))[0,1]))
except Exception:
    open('/kaggle/working/ERROR.txt','w').write(traceback.format_exc()); print(traceback.format_exc()); raise
